# Cell Cell Interaction Analysis

In [ ]:
import liana as li
import pseudobulk_analytics_utils_04 as pau
import scanpy as sc
import anndata as ad
import decoupler as dc
import gseapy as gp
import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl.utils import get_column_letter
import plotnine as p9
import omnipath as op
import graphviz
import corneto as cn
graphviz.Digraph()

%matplotlib inline

## A. Load in adata file and DESeq2 files from previous step 

In [ ]:
#### adata file with cell type annotations

adata_query_X =ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_MTG_Finalized.h5ad")

In [ ]:
#### DESeq2 files from previous steps

results_dir = "MTG_MMSE_pb_DEGs_low/"
de_results = {}

for f in glob.glob(os.path.join(results_dir, "*_deseq2_results.csv")):
    cell_type = os.path.basename(f).replace("_deseq2_results.csv", "").replace("_", " ")
    de_results[cell_type] = pd.read_csv(f, index_col=0)

In [ ]:
#### Confirm cell type names look ()

list(de_results.keys())[:5]          # confirm cell type names look right
de_results[list(de_results.keys())[0]].head()  # confirm gene symbols are the index, and columns include log2FoldChange/padj/stat

In [ ]:
#### Concatenate per-cell type DESeq2 results into one long combined dataframe

dea_df = pd.concat(
    [df.assign(cell_type=ct) for ct, df in de_results.items()]
).reset_index().rename(columns={'index': 'gene'})

In [ ]:
#### Group by cell type (predictions) AND resets gene to index and renames cell_type to predictions 
#### to match adata.obs, dea_df is the df containing ALL DESeq2 results, not the adata object 

groupby = 'predictions'  # must match your adata.obs column exactly

dea_df = dea_df.set_index('gene').rename(columns={'cell_type': groupby})

In [ ]:
#### Print percent of genes from dea_df in adata object

print(dea_df.index.isin(adata_query_X.var_names).mean()*100)

#### Print unaccounted for cell types in dea_df not found in adata object
#### Likely to do nomenclature issues ith "/" and "-"

print(set(dea_df[groupby].unique()) - set(adata_query_X.obs[groupby].unique()))

In [ ]:
#### Check possible values in adata obj that match the unaccounted for terms in dea_df

[c for c in adata_query_X.obs[groupby].unique() if 'L5' in c or 'L2' in c]


In [ ]:
#### Make mapping key to rename objects in dea_df by there adata obj names

dea_df[groupby] = dea_df[groupby].replace({'L5-6 NP': 'L5/6 NP', 'L2-3 IT': 'L2/3 IT'})


In [ ]:
#### Ensure that ALL cell types are accounted for (should print empty set!)

set(dea_df[groupby].unique()) - set(adata_query_X.obs[groupby].unique())


## B. Run LIANA pipeline

In [ ]:
### Run LIANA 

lr_res = li.multi.df_to_lr(adata_query_X,
                            dea_df=dea_df,
                            resource_name='consensus',
                            expr_prop=0.1,
                            groupby=groupby,
                            stat_keys=['stat', 'pvalue', 'padj'],
                            use_raw=False,
                            complex_col='stat',
                            verbose=True,
                            return_all_lrs=False,
                            )

In [ ]:
lr_res.shape
lr_res.head()

In [ ]:
#### Guage all column for 

lr_res.columns.tolist()

In [ ]:
#### Preliminary results table 

lr_res = lr_res.sort_values("interaction_stat", ascending=False, key=abs)

# significant interactions
sig_lr = lr_res[lr_res['interaction_padj'] < 0.05]
print(sig_lr.shape)
sig_lr.head(50)

In [ ]:
#### Make tileplot to guage top ligand and receptor activity through out each cell type
from plotnine import theme, element_text

p = li.pl.tileplot(liana_res=lr_res,
               fill='expr',
               label='padj',
               label_fun=lambda x: '*' if x < 0.05 else np.nan,
               top_n=15,
               orderby='interaction_stat',
               orderby_ascending=False,
               orderby_absolute=False,
               source_title='Ligand',
               target_title='Receptor',
               )

p = p + theme(figure_size=(18, 8))   # (width, height) in inches — widen as needed
p

In [ ]:
#### Make tileplot to guage communication quantity and direction

pair_counts = sig_lr.groupby(['source', 'target']).size().unstack(fill_value=0)

plt.figure(figsize=(10, 8))
sns.heatmap(pair_counts, annot=True, fmt='d', cmap='viridis', linewidths=0.5)
plt.title('Number of Significant LR Interactions by Source → Target')
plt.tight_layout()
plt.show()

In [ ]:
#### Assess specific source target interactions 


axis = sig_lr[(sig_lr['source']=='L2/3 IT') & (sig_lr['target']=='L4 IT')]
axis = axis.sort_values('interaction_stat', key=abs, ascending=False)
print(f"{len(axis)} significant interactions on L2/3 IT → L4 IT")
print(f"  up in low_MMSE:   {(axis['interaction_stat']>0).sum()}")
print(f"  down in low_MMSE: {(axis['interaction_stat']<0).sum()}")
print(axis[['ligand_complex','receptor_complex','interaction_stat','interaction_padj']].head(20).to_string())

In [ ]:
#### Check to ensure that ligand and receptors are not swapped


# Look at a few interactions where ligand vs receptor is biologically unambiguous
#check = sig_lr[sig_lr['ligand_complex'].isin(['NRG1','SEMA3C','SEMA4A','EFNB3','NXPH1'])]
#print(check[['source','ligand_complex','target','receptor_complex','interaction_stat']].head(20).to_string())

## C. Intracellular Signaling Networks

In [ ]:
#### utility function to select top n interactions

def select_top_n(d, n=None):
    d = dict(sorted(d.items(), key=lambda item: abs(item[1]), reverse=True))
    return {k: v for i, (k, v) in enumerate(d.items()) if i < n}

In [ ]:
#### Select source and target cell type of interest

source_label = 'L2/3 IT'
target_label = 'L2/3 IT'

lr_stats = sig_lr[sig_lr['source'].isin([source_label]) & sig_lr['target'].isin([target_label])].copy()
lr_stats = lr_stats.sort_values('interaction_stat', ascending=False, key=abs)

In [ ]:
#### Create receptor-interaction_stat dictionary

lr_dict = lr_stats.set_index('receptor')['interaction_stat'].to_dict()

#### Chooses the top n greatest changing receptors

input_scores = select_top_n(lr_dict, n=10)

In [ ]:
input_scores

In [ ]:
# First, let's transform the DEA statistics into a DF
# we will use these to estimate deregulated TF activity
dea_wide = dea_df[[groupby, 'stat']].reset_index(names='genes').pivot(index=groupby, columns='genes', values='stat')
dea_wide = dea_wide.fillna(0)
dea_wide

### C1. Utilize the CollecTRI regulon for network construction

In [ ]:
net = dc.op.collectri(organism='human', remove_complexes=False, license='academic', verbose=False)

In [ ]:
# Run Enrichment Analysis
estimates, pvals = dc.mt.ulm(data=dea_wide, net=net)
estimates.T.sort_values(target_label, key=abs, ascending=False).head()

In [ ]:
#### Select top n TFs whose activity is changing to eventually match with earlier selected receptors

tf_data = estimates.copy()
tf_dict = tf_data.loc[target_label].to_dict()


output_scores = select_top_n(tf_dict, n=5)

In [ ]:
#### Obtain ppi network
ppis = op.interactions.OmniPath().get(genesymbols = True)

ppis['mor'] = ppis['is_stimulation'].astype(int) - ppis['is_inhibition'].astype(int)
ppis = ppis[(ppis['mor'] != 0) & (ppis['curation_effort'] >= 5) & ppis['consensus_direction']] 

input_pkn = ppis[['source_genesymbol', 'mor', 'target_genesymbol']]
input_pkn.columns = ['source', 'mor', 'target']
input_pkn.head()

In [ ]:
#### Convert the PPI network into a knowledge graph
prior_graph = li.mt.build_prior_network(input_pkn, input_scores, output_scores, verbose=True)

In [ ]:
temp = adata_query_X[adata_query_X.obs[groupby] == target_label].copy()


In [ ]:
#### Create node weights for CORNETO

node_weights = pd.DataFrame(temp.X.getnnz(axis=0) / temp.n_obs, index=temp.var_names)
node_weights = node_weights.rename(columns={0: 'props'})
node_weights = node_weights['props'].to_dict()

In [ ]:
#### Reconstructs the signaling network that connects input receptors to  output TFs

df_res, problem = li.mt.find_causalnet(
    prior_graph, 
    input_scores, 
    output_scores, 
    node_weights,
    # penalize (max_penalty) nodes with counts in less than 0.1 of the cells
    node_cutoff=0.1, 
    max_penalty=1,
    # the penaly of those in > 0.1 prop of cells set to:
    min_penalty=0.01,
    edge_penalty=0.1,
    verbose=False,
    max_runs=50, # NOTE that this repeats the solving either until the max runs are reached
    stable_runs=10, # or until X number of consequitive stable runs are reached (i.e. no new edges are added)
    solver='SCIP' # 'scipy' is available by default, but often results in suboptimal solutions
    )

In [ ]:
#### Visualize network graph

cn.methods.carnival.visualize_network(df_res)

In [ ]:
#### Save network graph

graph = cn.methods.carnival.visualize_network(df_res)

# Save as PNG
graph.render(filename='Astrocyte_L2_3_IT_causal_network', format='png', cleanup=True)

# Or save as PDF (often cleaner for figures going into a thesis/paper)
graph.render(filename='Astrocyte_causal_L2_L3_IT_network', format='pdf', cleanup=True)

# Or save as SVG (scalable, good for editing later)
graph.render(filename='Astrocyte_causal_L2_L3_IT_network', format='svg', cleanup=True)

In [ ]:
print("target_label:", target_label) 

In [ ]:
print(df_res.columns.tolist())
print(df_res.head(20))

In [ ]:
help(cn.methods.carnival.visualize_network)

In [ ]:
import inspect
print(inspect.getsource(cn.methods.carnival.visualize_network))


In [ ]:
black = df_res[df_res['edge_pred_val'] == 0]
print(f"{len(black)} sign-neutral (black) edges out of {len(df_res)} total")
print(black[['source','source_type','target','target_type','edge_type']].to_string())

In [ ]:
print("red (positive flow):", (df_res['edge_pred_val'] > 0).sum())
print("blue (negative flow):", (df_res['edge_pred_val'] < 0).sum())
print("black (zero):", (df_res['edge_pred_val'] == 0).sum())

In [ ]:
print(df_res[df_res['edge_pred_val'] < 0][['source','edge_type','target','edge_pred_val']].to_string())


In [ ]:
# edges that render BLACK: int(edge_pred_val) == 0 but value isn't necessarily exactly 0
import numpy as np
black_mask = df_res['edge_pred_val'].apply(lambda x: int(x) == 0)
print(f"black-rendered edges: {black_mask.sum()}")
print(df_res[black_mask][['source','edge_type','target','edge_pred_val']].to_string())

# and see the full distribution of edge_pred_val values
print("\nunique edge_pred_val values:")
print(df_res['edge_pred_val'].value_counts())

In [ ]:
print("dtype:", df_res['edge_pred_val'].dtype)
print("types present:", df_res['edge_pred_val'].apply(type).value_counts())
# show a black one's actual value and what int() does to it
bad = df_res[df_res['edge_pred_val'].apply(lambda x: int(x)==0)].iloc[0]['edge_pred_val']
print("example black value:", repr(bad), "| int() =>", int(bad))

In [ ]:
edge_counts = df_res.groupby(['source','target']).size().sort_values(ascending=False)
print("max copies of any edge:", edge_counts.max())
print(edge_counts.head(10))